In [1]:
import pickle

In [2]:
xgb = pickle.load(open('../artifacts/xgboost_model.pkl', 'rb'))

In [3]:
rf = pickle.load(open('../artifacts/rf_model.pkl', 'rb'))

In [4]:
lr = pickle.load(open('../artifacts/logistic_model.pkl', 'rb'))

In [5]:
print("All models loaded successfully!")

All models loaded successfully!


In [6]:
import numpy as np

In [7]:
from sklearn.model_selection import StratifiedKFold

In [8]:
from sklearn.base import clone

In [9]:
import pickle

In [10]:
X_train, X_test, y_train, y_test = pickle.load(
    open('../artifacts/preprocessed_data.pkl', 'rb')
)

In [11]:
print(X_train.shape)

(1972, 52)


In [12]:
import numpy as np

oof_xgb = np.zeros(len(X_train))

In [13]:
# Create arrays for all base model predictions

oof_rf = np.zeros(len(X_train))
oof_lr = np.zeros(len(X_train))

In [14]:
# Import required libraries

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

In [15]:
# Create Stratified KFold object

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [16]:
# Start fold loop

for train_idx, valid_idx in skf.split(X_train, y_train):

    # Split training and validation data
    X_tr, X_val = X_train[train_idx], X_train[valid_idx]
    y_tr, y_val = y_train[train_idx], y_train[valid_idx]

    # Clone models
    xgb_fold = clone(xgb)
    rf_fold = clone(rf)
    lr_fold = clone(lr)

    # Train models
    xgb_fold.fit(X_tr, y_tr)
    rf_fold.fit(X_tr, y_tr)
    lr_fold.fit(X_tr, y_tr)

    # Generate probability predictions
    oof_xgb[valid_idx] = xgb_fold.predict_proba(X_val)[:, 1]
    oof_rf[valid_idx] = rf_fold.predict_proba(X_val)[:, 1]
    oof_lr[valid_idx] = lr_fold.predict_proba(X_val)[:, 1]

In [17]:
# Check first few predictions

print(oof_xgb[:5])
print(oof_rf[:5])
print(oof_lr[:5])

[0.00785142 0.12778264 0.02494389 0.24970798 0.0423693 ]
[0.14782238 0.31213572 0.1591854  0.25856298 0.17231561]
[0.00164513 0.44441893 0.01134741 0.64151417 0.82944005]


In [18]:
# Build Meta Feature Matrix for Training

X_meta_train = np.column_stack((
    oof_xgb,
    oof_rf,
    oof_lr
))

In [19]:
# Check shape of meta training data

print("Shape of X_meta_train:", X_meta_train.shape)

Shape of X_meta_train: (1972, 3)


In [20]:
# Generate predictions on test set

xgb_test_pred = xgb.predict_proba(X_test)[:, 1]
rf_test_pred = rf.predict_proba(X_test)[:, 1]
lr_test_pred = lr.predict_proba(X_test)[:, 1]

In [21]:
# Build Meta Feature Matrix for Test Data

X_meta_test = np.column_stack((
    xgb_test_pred,
    rf_test_pred,
    lr_test_pred
))

In [22]:
# Check shape of meta test data

print("Shape of X_meta_test:", X_meta_test.shape)

Shape of X_meta_test: (294, 3)


In [23]:
# Check first 5 rows of meta training data

print(X_meta_train[:5])

[[0.00785142 0.14782238 0.00164513]
 [0.12778264 0.31213572 0.44441893]
 [0.02494389 0.1591854  0.01134741]
 [0.24970798 0.25856298 0.64151417]
 [0.0423693  0.17231561 0.82944005]]


In [24]:
# Check first 5 rows of meta test data

print(X_meta_test[:5])

[[0.32235757 0.61410142 0.18186165]
 [0.00878334 0.11091667 0.01007332]
 [0.04672674 0.17763236 0.04118582]
 [0.00404307 0.04449075 0.02171442]
 [0.25014281 0.41710378 0.84187251]]


In [25]:
# Check for missing values

print(np.isnan(X_meta_train).sum())
print(np.isnan(X_meta_test).sum())

0
0


In [26]:
# Check minimum and maximum probabilities

print(X_meta_train.min())
print(X_meta_train.max())

3.5289742873827605e-05
0.9998020529747009


In [27]:
# Import Meta Learner

from sklearn.linear_model import LogisticRegression

In [28]:
# Create Meta Learner Model

meta_model = LogisticRegression(
    random_state=42
)

In [29]:
# Train Meta Learner

meta_model.fit(X_meta_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [30]:
# Generate Final Predictions

y_pred = meta_model.predict(X_meta_test)

In [31]:
# Generate Prediction Probabilities

y_pred_proba = meta_model.predict_proba(X_meta_test)[:, 1]

In [32]:
# Check first few predictions

print(y_pred[:10])

[1 0 0 0 0 0 0 0 0 1]


In [33]:
# Check first few probabilities

print(y_pred_proba[:10])

[0.53124081 0.02037981 0.03385526 0.01420495 0.31042354 0.19294582
 0.05498014 0.04198043 0.01408685 0.65778277]
